In [1]:
import pandas as pd

# 1. Load the MEDIC test file (we can just use the filename since the notebook is in the same folder)
tsv_path = "MEDIC_test.tsv"

print("Loading MEDIC test dataset...")
df = pd.read_csv(tsv_path, sep='\t')

print(f"Original test set size: {len(df)} rows")

# 2. Find the column that contains the image paths
image_column = [col for col in df.columns if 'image' in col.lower()][0]
print(f"Filtering based on column: '{image_column}'")

# 3. Filter OUT CrisisMMD data
# The ~ symbol means "NOT", keeping only rows that do not contain 'crisismmd'
df_ood = df[~df[image_column].str.contains('crisismmd', case=False, na=False)]

print(f"Filtered OOD test set size (AIDR/DMD only): {len(df_ood)} rows")

# 4. Save the clean OOD dataset
output_path = "MEDIC_test_OOD_only.tsv"
df_ood.to_csv(output_path, sep='\t', index=False)

print(f"\nSuccess! Clean OOD dataset saved to: {output_path}")

Loading MEDIC test dataset...
Original test set size: 15688 rows
Filtering based on column: 'image_id'
Filtered OOD test set size (AIDR/DMD only): 10846 rows

Success! Clean OOD dataset saved to: MEDIC_test_OOD_only.tsv


In [2]:
# Use the new clean dataset we just generated
dataset_path = "MEDIC_test_OOD_only.tsv" 
df_ood = pd.read_csv(dataset_path, sep='\t')

In [4]:
# Print every single column name in the dataset
print("All columns in the dataset:")
for col in df_ood.columns:
    print(f"- {col}")

All columns in the dataset:
- image_id
- event_name
- image_path
- damage_severity
- informative
- humanitarian
- disaster_types


In [5]:
# Print the unique labels in the humanitarian column
print("Unique Humanitarian Labels in MEDIC:")
print(df_ood['humanitarian'].unique())

Unique Humanitarian Labels in MEDIC:
['infrastructure_and_utility_damage' 'not_humanitarian'
 'affected_injured_or_dead_people'
 'rescue_volunteering_or_donation_effort']


In [6]:
import torch
import pandas as pd
from PIL import Image
from sklearn.metrics import classification_report
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running evaluation on: {device}")

# 1. Load the Clean OOD Dataset
dataset_path = "MEDIC_test_OOD_only.tsv" 
df_ood = pd.read_csv(dataset_path, sep='\t')

# Drop rows where the humanitarian label is completely missing
df_ood = df_ood.dropna(subset=['humanitarian'])

# 2. Label Mapping 
# (Note: I am guessing their format based on standard conventions. 
# If the output from Step 1 looks slightly different, just tweak the left side of this dictionary!)
label_mapping = {
    "affected_individuals": "affected_individuals",
    "infrastructure_and_utility_damage": "infrastructure_and_utility_damage",
    "rescue_volunteering_or_donation_effort": "rescue_volunteering_or_donation_effort",
    "other_relevant_information": "other_relevant_information",
    "not_humanitarian": "not_humanitarian"
}

# Apply the mapping
df_ood['ground_truth_mapped'] = df_ood['humanitarian'].map(label_mapping)

# Drop rows that didn't map to your 5 core categories
df_ood = df_ood.dropna(subset=['ground_truth_mapped'])
print(f"Testing on {len(df_ood)} valid OOD samples.\n")

# ... (The rest of your model loading and inference loop stays exactly the same!)

Running evaluation on: cuda
Testing on 10325 valid OOD samples.

